# Amazon Week 2 — Baseline Ranking System
**One-click Google Colab pipeline**

Roadmap implemented:
1. BM25 / TF-IDF lexical baselines
2. Fixed **query-level** train/test split
3. Ranking features
4. **XGBoost pointwise** baseline
5. NDCG@10 / Recall@10 / MRR
6. BM25 vs TF-IDF vs XGBoost benchmark
7. Feature importance + error analysis
8. Automatic ZIP export

> Evaluation is performed on the **Week 1 judged query-product candidate set**. This is deliberate: products without relevance judgments cannot be used as negatives in an offline ranking metric without making an unsupported assumption.

In [ ]:
#@title 1. Install dependencies
!pip -q install rank_bm25 xgboost pyarrow
print("Dependencies installed.")

In [ ]:
#@title 2. Upload Week 1 data
from google.colab import files
import io, os, zipfile, glob, shutil
import pandas as pd
import numpy as np

uploaded = files.upload()
WORK = "/content/amazon_week2"
OUT = f"{WORK}/results"
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

for name, blob in uploaded.items():
    path = f"{WORK}/{name}"
    with open(path, "wb") as f:
        f.write(blob)
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(f"{WORK}/week1")

candidates = (
    glob.glob(f"{WORK}/**/query_product_relevance.parquet", recursive=True) +
    glob.glob(f"{WORK}/**/query_product_relevance.csv", recursive=True)
)
if not candidates:
    raise FileNotFoundError(
        "Upload amazon_week1_v4_processed_data.zip or query_product_relevance.csv/parquet"
    )

src = candidates[0]
df = pd.read_parquet(src) if src.endswith(".parquet") else pd.read_csv(src)

required = ["query_id","query","product_id","title","relevance_label"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded:", src)
print("Shape:", df.shape)
print("Queries:", df["query_id"].nunique())
print("Label distribution:")
display(df["relevance_label"].value_counts().sort_index().rename("count").to_frame())

In [ ]:
#@title 3. Prepare data + fixed query-level split
from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.20

for c in ["query","title","description","category","brand"]:
    if c not in df.columns:
        df[c] = ""
    df[c] = df[c].fillna("").astype(str)

for c in ["price","average_rating","review_count"]:
    if c not in df.columns:
        df[c] = np.nan
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["relevance_label"] = pd.to_numeric(df["relevance_label"], errors="coerce").fillna(0).astype(int)

query_ids = np.array(sorted(df["query_id"].unique()))
train_q, test_q = train_test_split(query_ids, test_size=TEST_SIZE, random_state=SEED)

df["split"] = np.where(df["query_id"].isin(test_q), "test", "train")

pd.DataFrame({"query_id": train_q}).to_csv(f"{OUT}/fixed_train_queries.csv", index=False)
pd.DataFrame({"query_id": test_q}).to_csv(f"{OUT}/fixed_test_queries.csv", index=False)

assert set(train_q).isdisjoint(set(test_q))
print(f"Train queries: {len(train_q)}")
print(f"Test queries : {len(test_q)}")
print("Query leakage:", len(set(train_q) & set(test_q)))

In [ ]:
#@title 4. Build BM25 + TF-IDF features
import re
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

# Product document = title + description
df["document"] = (df["title"] + " " + df["description"]).str.strip()
df["bm25_score"] = 0.0
df["tfidf_similarity"] = 0.0

# Scores are calculated WITHIN each judged query candidate set.
# This gives a fair apples-to-apples offline ranking comparison.
for qid, idx in df.groupby("query_id").groups.items():
    idx = list(idx)
    q = df.loc[idx[0], "query"]
    docs = df.loc[idx, "document"].tolist()

    tokenized_docs = [tokenize(x) for x in docs]
    bm25 = BM25Okapi(tokenized_docs)
    df.loc[idx, "bm25_score"] = bm25.get_scores(tokenize(q))

    texts = [q] + docs
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=1)
    X = vec.fit_transform(texts)
    sims = cosine_similarity(X[0:1], X[1:]).ravel()
    df.loc[idx, "tfidf_similarity"] = sims

print("Lexical features complete.")
display(df[["query","title","relevance_label","bm25_score","tfidf_similarity"]].head(10))

In [ ]:
#@title 5. Create metadata / text features
def token_set(s):
    return set(tokenize(s))

def overlap_ratio(q, text):
    qset = token_set(q)
    if not qset:
        return 0.0
    return len(qset & token_set(text)) / len(qset)

df["query_len"] = df["query"].map(lambda x: len(tokenize(x)))
df["title_len"] = df["title"].map(lambda x: len(tokenize(x)))
df["description_len"] = df["description"].map(lambda x: len(tokenize(x)))
df["query_title_overlap"] = [overlap_ratio(q,t) for q,t in zip(df["query"],df["title"])]
df["query_description_overlap"] = [overlap_ratio(q,t) for q,t in zip(df["query"],df["description"])]

# Missing numeric values are imputed from TRAIN data only.
numeric_base = ["price","average_rating","review_count"]
train_mask = df["split"].eq("train")
for c in numeric_base:
    med = df.loc[train_mask, c].median()
    if pd.isna(med):
        med = 0.0
    df[c] = df[c].fillna(med)

# log transform review count while retaining raw signal
df["log_review_count"] = np.log1p(df["review_count"].clip(lower=0))

FEATURES = [
    "bm25_score",
    "tfidf_similarity",
    "price",
    "average_rating",
    "log_review_count",
    "query_len",
    "title_len",
    "description_len",
    "query_title_overlap",
    "query_description_overlap",
]

df.to_parquet(f"{OUT}/ranking_features.parquet", index=False)
print("Features:", FEATURES)

In [ ]:
#@title 6. Ranking evaluation functions
from sklearn.metrics import ndcg_score

def reciprocal_rank(labels):
    labels = np.asarray(labels)
    hits = np.where(labels > 0)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)

def recall_at_k(labels, k=10):
    labels = np.asarray(labels)
    relevant = (labels > 0)
    denom = relevant.sum()
    if denom == 0:
        return np.nan
    return relevant[:k].sum() / denom

def evaluate_ranking(frame, score_col, k=10):
    rows = []
    for qid, g in frame.groupby("query_id"):
        g = g.sort_values(score_col, ascending=False)
        y = g["relevance_label"].to_numpy()
        scores = g[score_col].to_numpy()

        # sklearn NDCG handles graded relevance labels directly.
        try:
            ndcg = ndcg_score([y], [scores], k=k)
        except Exception:
            ndcg = np.nan

        rows.append({
            "query_id": qid,
            "NDCG@10": ndcg,
            "Recall@10": recall_at_k(y, k),
            "MRR": reciprocal_rank(y),
            "n_candidates": len(g)
        })
    per_query = pd.DataFrame(rows)
    summary = per_query[["NDCG@10","Recall@10","MRR"]].mean()
    return summary, per_query

test_df = df[df["split"].eq("test")].copy()

bm25_summary, bm25_q = evaluate_ranking(test_df, "bm25_score")
tfidf_summary, tfidf_q = evaluate_ranking(test_df, "tfidf_similarity")

print("BM25")
display(bm25_summary.to_frame("value"))
print("TF-IDF")
display(tfidf_summary.to_frame("value"))

In [ ]:
#@title 7. Train XGBoost POINTWISE baseline
from xgboost import XGBRegressor

train_df = df[df["split"].eq("train")].copy()
test_df = df[df["split"].eq("test")].copy()

X_train = train_df[FEATURES]
y_train = train_df["relevance_label"]
X_test = test_df[FEATURES]

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=350,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1
)
model.fit(X_train, y_train)

test_df["xgb_score"] = model.predict(X_test)
xgb_summary, xgb_q = evaluate_ranking(test_df, "xgb_score")

pred_cols = ["query_id","query","product_id","title","relevance_label",
             "bm25_score","tfidf_similarity","xgb_score"]
test_df[pred_cols].to_csv(f"{OUT}/xgboost_predictions.csv", index=False)

print("XGBoost pointwise")
display(xgb_summary.to_frame("value"))

In [ ]:
#@title 8. Benchmark table + target check
benchmark = pd.DataFrame({
    "BM25": bm25_summary,
    "TF-IDF": tfidf_summary,
    "XGBoost Pointwise": xgb_summary
}).T.reset_index().rename(columns={"index":"model"})

bm25_ndcg = benchmark.loc[benchmark.model.eq("BM25"), "NDCG@10"].iloc[0]
xgb_ndcg = benchmark.loc[benchmark.model.eq("XGBoost Pointwise"), "NDCG@10"].iloc[0]
improvement = (xgb_ndcg / bm25_ndcg - 1) * 100 if bm25_ndcg != 0 else np.nan

benchmark["NDCG_vs_BM25_%"] = (
    (benchmark["NDCG@10"] / bm25_ndcg - 1) * 100
)
benchmark.to_csv(f"{OUT}/benchmark_metrics.csv", index=False)

print(f"XGBoost NDCG@10 improvement vs BM25: {improvement:.2f}%")
print("Roadmap target >= 5%:", "PASS" if improvement >= 5 else "NOT MET — report honestly")
display(benchmark.round(4))

In [ ]:
#@title 9. Top-10 ranking outputs
def topk_output(frame, score_col, model_name, k=10):
    out = (frame.sort_values(["query_id", score_col], ascending=[True,False])
                .groupby("query_id", group_keys=False)
                .head(k)
                .copy())
    out["rank"] = out.groupby("query_id").cumcount() + 1
    out["model"] = model_name
    return out[["model","query_id","query","rank","product_id","title",
                "relevance_label",score_col]]

bm25_top10 = topk_output(test_df, "bm25_score", "BM25")
tfidf_top10 = topk_output(test_df, "tfidf_similarity", "TF-IDF")
xgb_top10 = topk_output(test_df, "xgb_score", "XGBoost Pointwise")

bm25_top10.to_csv(f"{OUT}/bm25_top10.csv", index=False)
tfidf_top10.to_csv(f"{OUT}/tfidf_top10.csv", index=False)
xgb_top10.to_csv(f"{OUT}/xgboost_top10.csv", index=False)

print("Saved Top-10 outputs.")

In [ ]:
#@title 10. Error analysis
comparison = (
    bm25_q[["query_id","NDCG@10"]].rename(columns={"NDCG@10":"bm25_ndcg"})
    .merge(xgb_q[["query_id","NDCG@10"]].rename(columns={"NDCG@10":"xgb_ndcg"}), on="query_id")
)
comparison["delta_ndcg"] = comparison["xgb_ndcg"] - comparison["bm25_ndcg"]

query_lookup = test_df[["query_id","query"]].drop_duplicates()
comparison = comparison.merge(query_lookup, on="query_id", how="left")

comparison["case"] = np.select(
    [comparison["delta_ndcg"] >= 0.10, comparison["delta_ndcg"] <= -0.10],
    ["XGBoost wins", "BM25 wins"],
    default="Similar"
)
comparison = comparison.sort_values("delta_ndcg")
comparison.to_csv(f"{OUT}/error_analysis.csv", index=False)

print("Worst XGBoost cases:")
display(comparison.head(10)[["query","bm25_ndcg","xgb_ndcg","delta_ndcg","case"]])
print("Best XGBoost cases:")
display(comparison.tail(10).sort_values("delta_ndcg", ascending=False)
        [["query","bm25_ndcg","xgb_ndcg","delta_ndcg","case"]])

In [ ]:
#@title 11. Visualizations
import matplotlib.pyplot as plt

# Model comparison
plot_df = benchmark.set_index("model")[["NDCG@10","Recall@10","MRR"]]
ax = plot_df.plot(kind="bar", figsize=(10,5))
ax.set_title("Week 2 Baseline Ranking Benchmark")
ax.set_ylabel("Metric")
ax.set_ylim(0, min(1.05, max(1.0, plot_df.max().max()*1.15)))
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(f"{OUT}/model_comparison.png", dpi=180)
plt.show()

# Feature importance
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=True)

ax = importance.plot(x="feature", y="importance", kind="barh",
                     figsize=(9,6), legend=False)
ax.set_title("XGBoost Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{OUT}/feature_importance.png", dpi=180)
plt.show()

importance.sort_values("importance", ascending=False).to_csv(
    f"{OUT}/feature_importance.csv", index=False
)

In [ ]:
#@title 12. Generate Week 2 summary
best_model = benchmark.sort_values("NDCG@10", ascending=False).iloc[0]["model"]

summary = f'''
AMAZON SEARCH / LTR PROJECT — WEEK 2 BASELINE RANKING SYSTEM

DATA
- Query-product pairs: {len(df):,}
- Unique queries: {df["query_id"].nunique():,}
- Train queries: {len(train_q):,}
- Test queries: {len(test_q):,}
- Split strategy: fixed query-level split, seed={SEED}

MODELS
1. BM25 lexical baseline
2. TF-IDF cosine-similarity baseline
3. XGBoost pointwise regression baseline

FEATURES
{chr(10).join("- " + x for x in FEATURES)}

TEST METRICS
{benchmark.round(4).to_string(index=False)}

XGBOOST VS BM25
- NDCG@10 relative change: {improvement:.2f}%
- >=5% roadmap target: {"MET" if improvement >= 5 else "NOT MET"}
- Best model by NDCG@10: {best_model}

INTERPRETATION
BM25 is a lexical baseline because it ranks documents from query-term matching,
term frequency, inverse document frequency, and document-length normalization.
It does not directly learn how metadata signals should be weighted.

The XGBoost pointwise model can combine lexical relevance with product metadata
and text-overlap signals. It learns nonlinear relationships among those features
from Week 1 relevance labels, then produces a score used to rank products within
each query.

EVALUATION NOTE
Metrics are calculated on the judged Week 1 candidate set. Unjudged products are
not automatically treated as irrelevant.
'''.strip()

with open(f"{OUT}/week2_summary.txt","w") as f:
    f.write(summary)

print(summary)

In [ ]:
#@title 13. Export all Week 2 results
import shutil, os
zip_path = "/content/amazon_week2_baseline_ranking_results"
shutil.make_archive(zip_path, "zip", OUT)

print("Created:", zip_path + ".zip")
print("\nFiles:")
for x in sorted(os.listdir(OUT)):
    print(" -", x)

files.download(zip_path + ".zip")